<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px 30px 20px; border-radius: 10px; color: white; margin-bottom: 10px;">

# NexAU Cookbook — 构建企业问答 Agent

通过数据库场景学习如何编写 Skill，掌握 Skill 构建方法，最终搭建一个准确可用的企业问答 Agent。


</div>

### 核心概念

| 术语 | 含义 |
|:---|:---|
| **LLM**（大语言模型） | 理解你问题的 AI 模型，如 GPT-4、DeepSeek |
| **Tool**（工具） | Agent 可调用的函数，如 `execute_sql` |
| **Skill**（技能） | 教 Agent 理解数据的知识文件（SKILL.md） |
| **Agent**（智能体） | 自主行动的 LLM：调用工具 → 解读结果 → 回答你 |

### 教程路线图

| 章节 | 内容 |
|:---:|:---|
| **1** | 认识企业数据库 — 7 张表的结构和关系 |
| **2** | System Prompt — 先定义 Agent 的工作流 |
| **3** | 编写 SKILL.md — 让 Agent 正确回答的领域知识 |
| **4** | 自动生成 Skill — 表多时的快速起步方案 |
| **5** | 总结 |

---

### 准备工作

In [ ]:
import json
import re
import sqlite3
from pathlib import Path

---

### 1. 认识企业数据库

本教程使用一个**企业智能数据库**，包含 7 张表、50 家企业：

| 表名 | 说明 | 关联 |
|:---|:---|:---|
| `enterprise_basic` | 企业基本信息（注册、行业、专精特新等） | 主表，其他 enterprise_* 表通过 `credit_code` 关联 |
| `enterprise_contact` | 联系人（法人、总经理、联系人） | `credit_code` → `enterprise_basic` |
| `enterprise_financing` | 融资与上市（贷款、估值、上市状态） | `credit_code` → `enterprise_basic` |
| `enterprise_product` | 产品与知识产权（产品营收、专利） | `credit_code` → `enterprise_basic` |
| `industry` | 行业链节点（树结构：上游/中游/下游） | 被 `industry_enterprise` 引用 |
| `industry_enterprise` | 企业 ↔ 行业链映射 | `industry_id` → `industry`，`credit_code` → `enterprise_basic` |
| `users` | 平台用户账号 | **独立表，与企业无关** |

In [ ]:
DB_PATH = Path("enterprise.sqlite")
conn = sqlite3.connect(str(DB_PATH))

# 有哪些表？每张表多少行？
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
).fetchall()

for (t,) in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0]
    cols = conn.execute(f"PRAGMA table_info('{t}')").fetchall()
    print(f"{t:30s} {n:4d} rows, {len(cols):2d} columns")

看一下核心表 `enterprise_basic` 的关键字段：

In [ ]:
# 注意：register_capital 是 TEXT 类型 — 这是后面 Skill 要解决的头号陷阱
rows = conn.execute('''
    SELECT enterprise_name, register_district, register_capital, 
           enterprise_scale, zhuanjingtexin_level
    FROM enterprise_basic LIMIT 5
''').fetchall()

for r in rows:
    print(dict(zip(['企业名称', '注册区', '注册资本(万)', '规模', '专精特新'], r)))

conn.close()

---

### 2. System Prompt（系统提示词）

**System Prompt 是 Agent 的操作手册**，定义工作流和全局约束。我们先写它，再写 Skill。

```
规划 → 读 Skill → 写 SQL → 执行 → 反思 → 回答
```

> **Skill vs System Prompt 的分工：** Skill 存放每张表的详细知识（动态加载），System Prompt 定义工作流和全局约束（始终可见）。

In [ ]:
SYSTEM_PROMPT = '''
You are an enterprise data agent for the North Nova enterprise intelligence
database — a SQLite mirror of seven core tables describing Chinese enterprises,
their contacts, financing status, products, and the industry chains they belong to.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: `enterprise_basic`, `enterprise_contact`, `enterprise_financing`,
  `enterprise_product`, `industry`, `industry_enterprise`, `users`
- Primary join key across enterprise_* tables: `credit_code`

## Table Knowledge (Skills)

Detailed schema, common values, and example queries for each table are
provided as Skills. ALWAYS read the relevant Skill before writing a query.

### Key gotchas (always visible)

- `enterprise_basic.register_capital` is TEXT — use CAST(register_capital AS REAL)
- `enterprise_product.daily_capacity` is TEXT — use CAST(daily_capacity AS REAL)
- `zhuanjingtexin_level` values: 专精特新中小企业 < 专精特新潜在"小巨人"企业 < 专精特新"小巨人"企业
- `users` table = platform accounts, NOT enterprises. "用户" usually means enterprise.
- `industry.chain_position` ('up'/'mid'/'down') only on depth=1 nodes

## Workflow

1. **Plan.** Identify which tables you need.
2. **Read Skills.** For every table you will touch, read its Skill first.
3. **Write the SQL.** SQLite syntax. Always LIMIT. Prefer explicit columns.
4. **Execute.** Call `execute_sql`.
5. **Reflect.** If 0 rows or warnings, re-read the Skill and try again.
6. **Answer** in the user\'s language. Be concise. Show key data + SQL.

## Constraints

- Only SELECT queries work — the tool rejects everything else.
- Don\'t hallucinate columns. If unsure, query the schema first.
- Mock data: names like 测试企业_N, credit codes like MOCKCREDIT0000000001.
'''

print(SYSTEM_PROMPT)

---

### 3. 编写 SKILL.md — 提升问答准确率的关键

基础 Agent 经常答错——不是因为模型"不会写 SQL"，而是**不了解这个数据库的具体情况**。

`register_capital` 是 TEXT 类型，直接 `ORDER BY` 会按字符串排序；`zhuanjingtexin_level` 有隐式的等级高低；`users` 表和企业数据无关——这些"潜规则"模型不可能自己知道。

**Skill 就是把你对数据库的了解传递给模型的方式。**

#### 3.1 Skill 的基本结构

Skill 就是一个文件夹里放一个 `SKILL.md`。文件由两部分组成：

```markdown
---
name: enterprise_basic
description: Use this skill whenever the user asks about an enterprise's
  identity, registration, location, scale, or industry classification.
---

# enterprise_basic — 企业基本信息
（正文：Schema、示例 SQL、注意事项...）
```

- **`description`**（始终对模型可见）——告诉模型"什么时候该读取这个 Skill"
- **正文**（模型决定读取时才加载）——告诉模型"这张表具体怎么查"

接下来的重点：**正文里写什么、怎么写，才能让 Agent 答对？**

#### 3.2 第一步：审计你的数据库

编写 Skill 之前，逐表回答以下问题：

| 问题 | 目的 |
|:---|:---|
| 这张表存的是什么？一行代表什么？ | 让模型理解业务含义 |
| 哪些列的存储类型和业务含义不匹配？ | 发现类型陷阱（TEXT 存数字） |
| 哪些列只有几个固定取值？ | 列出枚举值，减少猜测 |
| 聚合时需要排除哪些数据？ | 明确业务过滤规则 |
| 哪些字段是预计算的？ | 防止模型重复计算 |
| 表之间怎么 join？ | 标注主键和外键 |
| 用户最常问什么？正确的 SQL 怎么写？ | 准备示例查询 |

**每回答一个问题，就往 SKILL.md 里写一条。**

以 `enterprise_basic` 为例：
- 一行 = 一家企业，37 列涵盖注册信息、行业分类、专精特新等级
- `register_capital` 是 TEXT 存数字 → 需要 CAST 提示
- `enterprise_scale` 只有 4 个值（微型/小型/中型/大型）→ 列出枚举
- `zhuanjingtexin_level` 有 3 个等级 + NULL → 列出并标注高低顺序
- 通过 `credit_code` 与其他 enterprise_* 表 join

#### 3.3 类型陷阱：准确率的第一杀手

`register_capital` 是 TEXT，直接 `ORDER BY register_capital DESC` 结果完全错误。

**Skill 中需要在三个地方提示模型**：

**① Schema 表——紧邻列名标注**：
```
| `register_capital` | TEXT | 注册资本（万元）— **TEXT not numeric**, use `CAST(register_capital AS REAL)` |
```

**② Example queries——展示正确写法**：
```sql
SELECT enterprise_name, CAST(register_capital AS REAL) AS capital_wan
FROM enterprise_basic
WHERE register_district = '海淀区'
ORDER BY capital_wan DESC LIMIT 10;
```

**③ Gotchas——解释为什么错**：
```
- `register_capital` is TEXT — always CAST(register_capital AS REAL).
  Direct ORDER BY gives wrong results (string sort: "8000" > "50000").
```

三处信息互相加强：Schema 让模型注意到类型，Example 提供正确模板，Gotchas 解释原因。

同理，`enterprise_product.daily_capacity` 也是 TEXT 存数字，需要同样处理。

#### 3.4 业务规则：只有你知道的知识

类型陷阱看 Schema 就能发现，但**业务规则只有了解业务的人才知道**。这是 Skill 最有价值的部分。

**枚举等级**——`zhuanjingtexin_level` 的值有隐含的高低之分：
```
- zhuanjingtexin_level hierarchy:
  专精特新中小企业 < 专精特新潜在"小巨人"企业 < 专精特新"小巨人"企业
  NULL = 无专精特新认定
  "小巨人企业": WHERE zhuanjingtexin_level = '专精特新"小巨人"企业'
```
不写这条，Agent 不知道"小巨人"对应的精确值。

**上市状态过滤**——`enterprise_financing.listing_status` 有 4 个值：
```
- listing_status values: 未上市, 新三板, 已上市, 拟上市
  "已上市的企业" = WHERE listing_status = '已上市'
  "有上市计划的" = WHERE listing_status IN ('拟上市', '已上市')
```

**行业链的树结构**——`industry` 表是一棵树：
```
- depth=0: chain root, depth=1: 上游/中游/下游, depth=2: leaf nodes
- chain_position ('up'/'mid'/'down') ONLY on depth=1 nodes
- "AI 上游企业" = join industry (chain_position='up') → industry_enterprise → enterprise_basic
```

**容易混淆的表**——`users` 是平台账号，不是企业：
```
- "用户"在本数据库中通常指企业（enterprise_basic），不是平台用户（users）
- 只有明确问"平台管理员"、"登录账号"时才查 users 表
```

#### 3.5 跨表查询：引导模型正确 Join

用户问"专精特新小巨人企业中有哪些已上市？"需要 join `enterprise_basic` 和 `enterprise_financing` 两张表。

**在 description 中标注 join 关系**：
```yaml
description: >-
  Use this skill whenever the user asks about an enterprise's identity,
  registration, location, scale, or industry classification.
  Join other enterprise_* tables via credit_code.
```

**在 Schema 中标注 join key**：
```
| `credit_code` | TEXT | **Join key.** 统一社会信用代码 — shared across all enterprise_* tables. |
```

**在 Example queries 中提供 join 示例**（同时展示 join 条件和业务过滤）：
```sql
SELECT b.enterprise_name, f.listing_status, f.stock_code
FROM enterprise_basic b
JOIN enterprise_financing f ON b.credit_code = f.credit_code
WHERE b.zhuanjingtexin_level = '专精特新"小巨人"企业'
  AND f.listing_status = '已上市';
```

**三表 join 更需要引导**——"AI 上游有哪些企业"涉及 industry → industry_enterprise → enterprise_basic：
```sql
SELECT b.enterprise_name, i.name AS industry_node
FROM industry i
JOIN industry_enterprise ie ON ie.industry_id = i.id
JOIN enterprise_basic b ON b.credit_code = ie.credit_code
WHERE i.chain_id = 45 AND i.chain_position = 'up';
```

#### 3.6 Example queries 的选择策略

Example queries 是模型编写 SQL 的直接模板。

**选择原则**：
1. 覆盖最常见的查询模式（聚合、排序、分组、join）
2. 每个示例至少展示一个"坑"的正确处理（CAST、枚举值、join 路径）
3. 选择用户真的会问的问题

**每张表 2-4 个示例**——太少模型缺少模板，太多模型可能忽略。

| 好示例 | 差示例 |
|:---|:---|
| 涉及 CAST 的聚合查询 | `SELECT * FROM table LIMIT 10` |
| 带枚举值过滤的分组统计 | 不带 WHERE 的全表查询 |
| 多表 join + 业务过滤 | 单表单列查询 |

差示例的问题：模型已经会写简单查询了。**示例要展示的是模型自己不知道的东西。**

#### 3.7 description：让模型找到正确的 Skill

`description` 决定模型"是否读取这个 Skill"。它始终可见，所以写法关键：

**正面路由**——关键词要覆盖用户的不同说法：
```yaml
description: >-
  Use this skill whenever the user asks about an enterprise's financing —
  bank loans, equity rounds, valuation, listing status, planned listing
  location, or future financing demand. Join with enterprise_basic via credit_code.
```

**负面路由**——有些表容易被混淆，需要主动"劝退"：
```yaml
description: >-
  Use this skill ONLY when the user explicitly asks about platform users —
  login accounts, SSO ids, roles. This table is unrelated to the enterprise
  tables. Most questions about "用户" actually mean enterprises, not platform users.
```

| 反模式 | 问题 | 改进 |
|:---|:---|:---|
| `"企业基本信息"` | 太笼统 | `"Use when user asks about registration, location, scale, industry, 专精特新 status"` |
| `"融资表"` | 纯标签 | `"Use when user asks about bank loans, equity, valuation, listing status"` |
| 把 Schema 塞进 description | 浪费 token | description 只放路由提示，Schema 放正文 |

#### 3.8 完整示例：核心表的 SKILL.md

下面展示 `enterprise_basic` 和 `users` 两个关键 Skill，注意观察如何运用前面讲到的技巧。

In [ ]:
# ===== enterprise_basic（最核心的表）=====
enterprise_basic_skill = '''---
name: enterprise_basic
description: >-
  Use this skill whenever the user asks about an enterprise's identity,
  registration, location, scale, industry classification, or 专精特新 status.
  This is the primary table — almost every query about a company starts here.
  Join other enterprise_* tables to it via credit_code.
---

# enterprise_basic — 企业基本信息

The central registry of enterprises. One row per enterprise, keyed by `credit_code`.

## When to use

- "Where is company X registered?" / "What district?"
- "How many small enterprises are there in 海淀区?"
- "List all 专精特新小巨人 enterprises"
- "What is the registered capital of …"

For contact info go to `enterprise_contact`, for financing go to `enterprise_financing`,
for products go to `enterprise_product`.

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Internal row id |
| `credit_code` | TEXT | **Join key.** 统一社会信用代码 — shared across all enterprise_* tables |
| `enterprise_name` | TEXT | 企业名称 |
| `register_district` | TEXT | 注册地所在区 (e.g. `海淀区`, `黄浦区`, `南山区`) |
| `register_capital` | TEXT | 注册资本（万元）— **TEXT not numeric**, use `CAST(register_capital AS REAL)` |
| `enterprise_scale` | TEXT | One of `微型`, `小型`, `中型`, `大型` |
| `enterprise_type` | TEXT | One of `民营`, `国有`, `合资`, `外资` |
| `industry_level1` | TEXT | 行业一级分类 (e.g. `制造业`, `金融业`) |
| `zhuanjingtexin_level` | TEXT | 专精特新等级 — see Common values |
| ... | | (37 columns total, see full schema for details) |

## Common values

- `enterprise_scale`: `微型`, `小型`, `中型`, `大型`
- `enterprise_type`: `民营`, `国有`, `合资`, `外资`
- `zhuanjingtexin_level`: `专精特新中小企业` < `专精特新潜在"小巨人"企业` < `专精特新"小巨人"企业`

## Example queries

**Top 10 enterprises by registered capital in 海淀区:**
```sql
SELECT enterprise_name,
       CAST(register_capital AS REAL) AS capital_wan,
       enterprise_scale
FROM enterprise_basic
WHERE register_district = '海淀区'
ORDER BY capital_wan DESC LIMIT 10;
```

**Count of 专精特新 enterprises by level:**
```sql
SELECT zhuanjingtexin_level, COUNT(*) AS n
FROM enterprise_basic
WHERE zhuanjingtexin_level IS NOT NULL
GROUP BY zhuanjingtexin_level ORDER BY n DESC;
```

**Join with financing to find listed 小巨人 companies:**
```sql
SELECT b.enterprise_name, f.listing_status, f.stock_code
FROM enterprise_basic b
JOIN enterprise_financing f ON b.credit_code = f.credit_code
WHERE b.zhuanjingtexin_level = '专精特新"小巨人"企业'
  AND f.listing_status = '已上市';
```

## Gotchas

- `register_capital` is **TEXT** — cast with `CAST(register_capital AS REAL)` for sorting/comparison.
- `zhuanjingtexin_level` contains Chinese quotes — match exactly: `'专精特新"小巨人"企业'`
- `enterprise_name` in mock data looks like `测试企业_N`.
'''

# ===== users（负面路由示例）=====
users_skill = '''---
name: users
description: >-
  Use this skill ONLY when the user explicitly asks about platform users —
  login accounts, SSO ids, roles. This table is unrelated to the enterprise
  tables and should not be joined to them. Most questions about "用户" actually
  mean enterprises, not platform users — confirm with the user if ambiguous.
---

# users — 平台用户账号

System users of the data platform itself — not enterprises.

## When to use

- "How many platform admins are there?"
- "List all users with the admin role"

**Do NOT use** when the user asks about enterprises, customers, or contacts.

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Internal id |
| `role` | TEXT | One of `user`, `admin` |
| `email` | TEXT | 邮箱 |

## Example queries

```sql
SELECT role, COUNT(*) AS n FROM users GROUP BY role;
```
'''

print(f'enterprise_basic_skill: {len(enterprise_basic_skill)} chars')
print(f'users_skill:            {len(users_skill)} chars')

#### 3.9 测试与迭代

写完 Skill 后，用以下方法验证：

1. **用 "When to use" 中的问题测试**——模型是否读取了正确的 Skill？SQL 是否正确？
2. **刻意测试边界情况**：
   - TEXT 数字列的排序："注册资本最高的企业"
   - 枚举值精确匹配："专精特新小巨人企业有哪些"
   - 跨表查询："已上市的小巨人企业"
   - 三表 join："AI 产业链上游有哪些企业"
   - 负面路由："平台有多少管理员"（应走 users，不是 enterprise_basic）
3. **每次 Agent 答错，分析根因并更新 Skill**：

| Agent 的错误 | Skill 如何修改 |
|:---|:---|
| 没有读取该 Skill | description 中加入用户使用的关键词 |
| 类型处理错误 | Schema + Gotchas + Example 三处同时加提示 |
| 枚举值写错 | Common values 中加入精确值 |
| Join 写错 | Schema 标注 FK，description 加 join 说明 |
| 误用 users 表 | users 的 description 加强负面路由 |

**Skill 是活文档——每修复一个错误就加一条提示，准确率逐步提升。**

---

### 4. 自动生成 Skill

7 张表手写没问题，70 张表就需要自动化了。下面的脚本读取任意 SQLite，为每张表生成 SKILL.md 草稿：

- 表结构（列名、类型、主键、外键）
- TEXT 列实际存储数字的情况
- 枚举型列（取值较少的列）
- 外键关系（自动生成 JOIN 示例）

需要人工判断的地方标记了 `[TODO]`。

#### 4.1 辅助函数

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False

#### 4.2 主生成函数

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "(no explicit PK)"

    fk_notes = [f"`{lc}` -> `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK -> `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " | ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            val_str = ", ".join(f"`{v[0]}`" for v in vals)
            common_sections.append(f"- `{col['name']}`: {val_str}")

    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** | PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

#### 4.3 在企业数据库上运行

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"Found {len(tables)} tables: {', '.join(tables)}\n")

for table in tables[:3]:  # 只展示前 3 张表，避免输出过长
    print("=" * 60)
    print(f"  {table}")
    print("=" * 60)
    print(generate_skill_md(conn, table))

conn.close()

看看自动生成器发现了什么：

- `enterprise_basic.register_capital` 被标记为"TEXT 存储数字" — 头号陷阱
- 枚举列（`enterprise_scale`、`enterprise_type`、`listing_status`）的取值被列出
- `[TODO]` 标记了需要你补充业务上下文的地方——比如专精特新等级的高低关系、跨表 join 路径

**自动生成是起点，人工补充业务知识才是关键。**

---

### 5. 总结

构建一个准确的企业问答 Agent，关键在于向模型传递领域知识：

**核心要点**：

- **Skill 的价值在于传递领域知识**——类型陷阱、业务规则、join 关系、枚举值
- **三处联动提示**——Schema 标注 + Example queries + Gotchas，确保模型不遗漏
- **description 是路由标签**——正面路由覆盖关键词，负面路由防止误用
- **Skill 是活文档**——每次答错就加一条提示，准确率逐步提升
- **System Prompt 只放工作流**——表结构知识放 Skill，保持 prompt 精简

**完整的目录结构**：

```
my_agent/
├── agent.yaml                        # Agent 配置
├── system_prompt.md                  # 工作流 + 全局约束
└── skills/
    ├── enterprise_basic/SKILL.md     # 企业基本信息
    ├── enterprise_contact/SKILL.md   # 联系人
    ├── enterprise_financing/SKILL.md # 融资与上市
    ├── enterprise_product/SKILL.md   # 产品与专利
    ├── industry/SKILL.md             # 行业链节点
    ├── industry_enterprise/SKILL.md  # 企业-行业映射
    └── users/SKILL.md                # 平台用户（负面路由）
```